<a href="https://colab.research.google.com/github/sheon1206/Weekly_Reading_Review/blob/main/daily_voca_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 구글 드라이브 마운트 (파일을 읽고 쓰기 위해 필요)
from google.colab import drive
drive.mount('/content/drive')

# 2. PDF 생성을 위한 reportlab 라이브러리 설치
!pip install reportlab

# 3. 한글 폰트(나눔고딕) 설치 (우분투 리눅스 환경)
!sudo apt-get install -y fonts-nanum

print("✅ 환경 설정이 완료되었습니다. 다음 코드를 실행하세요.")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 62.0 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 1 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 1s (7,033 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open st

In [ ]:
# [VOCA] PDF 생성 메인 코드 (레이아웃 최종 수정본)
import json
import os
import math
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import mm
from reportlab.platypus import (
    BaseDocTemplate, Frame, PageTemplate,
    Paragraph, Spacer, Table, TableStyle,
    PageBreak, FrameBreak, NextPageTemplate, KeepTogether
)
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ==========================================
# [1] 기본 설정
# ==========================================
BASE_DIR = '/content/drive/MyDrive/word_master_advanced'
INPUT_FILE = os.path.join(BASE_DIR, 'daily_voca_test.json')
OUTPUT_FILE = os.path.join(BASE_DIR, 'Daily_Voca_Test.pdf')

try:
    font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
    font_path_bold = '/usr/share/fonts/truetype/nanum/NanumGothicBold.ttf'
    if os.path.exists(font_path):
        pdfmetrics.registerFont(TTFont('NanumGothic', font_path))
        pdfmetrics.registerFont(TTFont('NanumGothicBold', font_path_bold))
    else:
        print("⚠️ 나눔고딕 폰트가 없습니다. Colab에서 !sudo apt-get install -y fonts-nanum 실행 필요")
except Exception as e:
    print(f"폰트 등록 에러: {e}")

# ==========================================
# [2] 스타일 정의
# ==========================================
def get_styles():
    styles = getSampleStyleSheet()
    font_name = 'NanumGothic'
    font_bold = 'NanumGothicBold'

    styles.add(ParagraphStyle(
        name='MainTitle', parent=styles['Heading1'],
        fontName=font_bold, fontSize=20,
        alignment=TA_CENTER, spaceAfter=15
    ))
    styles.add(ParagraphStyle(
        name='SectionTitle', parent=styles['Heading2'],
        fontName=font_bold, fontSize=14,
        textColor=colors.darkblue, spaceBefore=10, spaceAfter=5
    ))
    styles.add(ParagraphStyle(
        name='Instruction', parent=styles['Normal'],
        fontName=font_bold, fontSize=11, leading=14,
        textColor=colors.black, spaceBefore=5, spaceAfter=4,
        borderPadding=3, backColor=colors.whitesmoke
    ))
    styles.add(ParagraphStyle(
        name='QuestionContent', parent=styles['Normal'],
        fontName=font_name, fontSize=11, leading=16, spaceAfter=2
    ))
    styles.add(ParagraphStyle(
        name='HintText', parent=styles['Normal'],
        fontName=font_name, fontSize=9, leading=12,
        textColor=colors.gray, leftIndent=10
    ))
    styles.add(ParagraphStyle(
        name='AnswerText', parent=styles['Normal'],
        fontName=font_name, fontSize=10.5, leading=15
    ))
    styles.add(ParagraphStyle(
        name='DetailExp', parent=styles['Normal'],
        fontName=font_name, fontSize=9, leading=12,
        textColor=colors.darkslategray
    ))
    styles.add(ParagraphStyle(
        name='WordBankTitle', parent=styles['Normal'],
        fontName=font_bold, fontSize=11, leading=14,
        alignment=TA_LEFT, spaceAfter=3
    ))

    return styles

# ==========================================
# [3] PDF 생성 로직
# ==========================================
def create_voca_pdf():
    if not os.path.exists(INPUT_FILE):
        print(f"❌ 파일을 찾을 수 없습니다: {INPUT_FILE}")
        return

    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            raw = f.read().strip()
            if raw.startswith('```json'): raw = raw[7:]
            if raw.endswith('```'): raw = raw[:-3]
            data = json.loads(raw)
    except Exception as e:
        print(f"JSON 로드 에러: {e}")
        return

    doc = BaseDocTemplate(OUTPUT_FILE, pagesize=A4,
                          rightMargin=15*mm, leftMargin=15*mm,
                          topMargin=15*mm, bottomMargin=15*mm)

    page_w, page_h = A4
    margin = 15*mm
    gap = 8*mm
    col_w = (page_w - (2 * margin) - gap) / 2

    # -- 프레임 정의 --
    title_area_h = 35*mm
    content_h = page_h - 2*margin - title_area_h - 5*mm

    frame_title = Frame(margin, margin + content_h + 5*mm, page_w - 2*margin, title_area_h, id='title', showBoundary=0)
    frame_c1 = Frame(margin, margin, col_w, content_h, id='col1', showBoundary=0)
    frame_c2 = Frame(margin + col_w + gap, margin, col_w, content_h, id='col2', showBoundary=0)

    h_full = page_h - 2*margin
    frame_full_c1 = Frame(margin, margin, col_w, h_full, id='f_col1', showBoundary=0)
    frame_full_c2 = Frame(margin + col_w + gap, margin, col_w, h_full, id='f_col2', showBoundary=0)

    ans_title_h = 20*mm
    h_ans = page_h - 2*margin - ans_title_h - 5*mm
    frame_ans_title = Frame(margin, margin + h_ans + 5*mm, page_w - 2*margin, ans_title_h, id='ans_title', showBoundary=0)
    frame_ans_c1 = Frame(margin, margin, col_w, h_ans, id='ans_col1', showBoundary=0)
    frame_ans_c2 = Frame(margin + col_w + gap, margin, col_w, h_ans, id='ans_col2', showBoundary=0)

    doc.addPageTemplates([
        PageTemplate(id='FirstPage', frames=[frame_title, frame_c1, frame_c2]),
        PageTemplate(id='TwoColumnPage', frames=[frame_full_c1, frame_full_c2]),
        PageTemplate(id='AnswerStartPage', frames=[frame_ans_title, frame_ans_c1, frame_ans_c2]),
    ])

    story = []
    styles = get_styles()

    # 메타데이터
    meta = data.get('test_metadata', {})
    title_text = meta.get('title', 'Vocabulary Test')
    difficulty = meta.get('difficulty', 'General')
    questions = data.get('questions', [])

    # ====================
    # [문제지 파트]
    # ====================
    # 1. 상단 타이틀 (1단 전체 영역)
    story.append(Paragraph(title_text, styles['MainTitle']))

    info_data = [[f"Difficulty: {difficulty}", f"Name: _____________  Score: ______ / {len(questions)}"]]
    tbl_info = Table(info_data, colWidths=[col_w, col_w])
    tbl_info.setStyle(TableStyle([
        ('FONTNAME', (0,0), (-1,-1), 'NanumGothic'),
        ('LINEBELOW', (0,0), (-1,-1), 0.5, colors.black),
        ('ALIGN', (1,0), (1,0), 'RIGHT'),
        ('BOTTOMPADDING', (0,0), (-1,-1), 5)
    ]))
    story.append(tbl_info)

    # 타이틀 영역 끝내고 2단 컬럼(왼쪽)으로 이동
    story.append(FrameBreak())
    story.append(NextPageTemplate('TwoColumnPage'))

    # 2. Word Bank (왼쪽 컬럼 상단에 배치)
    if data.get('has_word_bank') and data.get('word_bank'):
        wb_list = data['word_bank']

        # Word Bank 타이틀
        story.append(Paragraph("<b>&lt;Word Bank&gt;</b>", styles['WordBankTitle']))

        # 2단 컬럼 폭에 맞춰야 하므로 col_w 사용
        # 한 줄에 3개 정도가 적당 (폭이 좁으므로)
        cols_per_row = 3
        # col_w는 mm 단위가 아니라 포인트 단위 실수이므로 그대로 사용
        cell_width = col_w / cols_per_row

        table_data = []
        row_data = []
        for word in wb_list:
            row_data.append(word)
            if len(row_data) == cols_per_row:
                table_data.append(row_data)
                row_data = []
        if row_data:
            while len(row_data) < cols_per_row:
                row_data.append("")
            table_data.append(row_data)

        wb_table = Table(table_data, colWidths=[cell_width] * cols_per_row)
        wb_table.setStyle(TableStyle([
            ('FONTNAME', (0,0), (-1,-1), 'NanumGothic'),
            ('FONTSIZE', (0,0), (-1,-1), 9),
            ('ALIGN', (0,0), (-1,-1), 'CENTER'),
            ('VALIGN', (0,0), (-1,-1), 'MIDDLE'),
            ('BOX', (0,0), (-1,-1), 1, colors.black),
            ('INNERGRID', (0,0), (-1,-1), 0.25, colors.gray),
            ('BACKGROUND', (0,0), (-1,-1), colors.whitesmoke),
            ('TOPPADDING', (0,0), (-1,-1), 4),
            ('BOTTOMPADDING', (0,0), (-1,-1), 4),
        ]))
        story.append(wb_table)
        story.append(Spacer(1, 5*mm)) # 보기와 문제 사이 간격

        # [중요] 여기 있던 FrameBreak()를 삭제했습니다.
        # 이제 코드는 자연스럽게 아래로 흐릅니다.

    # 3. 문제 루프
    last_instruction = ""
    for q in questions:
        block = []
        curr_instruction = q.get('instruction', 'Answer the question.')

        # 지시문
        if curr_instruction != last_instruction:
            if last_instruction != "":
                block.append(Spacer(1, 3*mm))
            block.append(Paragraph(f"■ {curr_instruction}", styles['Instruction']))
            last_instruction = curr_instruction

        # 문제
        q_text = f"<b>{q['id']}.</b> {q.get('content', '')}"
        block.append(Paragraph(q_text, styles['QuestionContent']))

        # 힌트
        #if q.get('hint'):
        #    block.append(Paragraph(f"└ Hint: {q['hint']}", styles['HintText']))

        block.append(Spacer(1, 4*mm))
        story.append(KeepTogether(block))

    # ====================
    # [정답지 파트]
    # ====================
    story.append(PageBreak())
    story.append(NextPageTemplate('AnswerStartPage'))

    story.append(Paragraph("Answer Key & Explanations", styles['MainTitle']))
    #story.append(FrameBreak())

    story.append(NextPageTemplate('TwoColumnPage'))
    story.append(Paragraph("■ Answers", styles['SectionTitle']))

    for q in questions:
        ans_block = []
        ans_text = f"<b>{q['id']}. {q.get('answer', '')}</b>"
        ans_block.append(Paragraph(ans_text, styles['AnswerText']))

        ctx_text = f"<font color='gray' size='8'>({q.get('content', '')})</font>"
        ans_block.append(Paragraph(ctx_text, styles['DetailExp']))

        ans_block.append(Spacer(1, 2*mm))
        story.append(KeepTogether(ans_block))

    try:
        doc.build(story)
        print(f"🎉 PDF 생성 완료! 경로: {OUTPUT_FILE}")
    except Exception as e:
        print(f"❌ PDF 생성 실패: {e}")

if __name__ == "__main__":
    create_voca_pdf()

🎉 PDF 생성 완료! 경로: /content/drive/MyDrive/word_master_advanced/Daily_Voca_Test.pdf
